# 02 The Data Model

This notebook audits the major record types exposed by TorchLens 2.0. An `Op` is one captured tensor-producing operation; a `Layer` is the stable graph position that can aggregate ops; modules, params, buffers, and grad functions provide the rest of the execution anatomy.

The example model includes parameters, a buffer, and autograd metadata. We capture with gradient saving enabled, then explicitly log backward metadata.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import torch
from torch import nn
import torchlens as tl

torch.manual_seed(13)


class AnatomyNet(nn.Module):
    """Tiny module that exposes params, a buffer, and backward metadata."""

    def __init__(self) -> None:
        """Initialize layers and a buffer."""
        super().__init__()
        self.proj = nn.Linear(3, 4)
        self.norm = nn.LayerNorm(4)
        self.head = nn.Linear(4, 1)
        self.register_buffer("bias_shift", torch.tensor([0.1]))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run one scalar-producing forward pass."""
        h = self.norm(torch.relu(self.proj(x)))
        return self.head(h) + self.bias_shift


model = AnatomyNet().eval()
x = torch.randn(2, 3, requires_grad=True)
trace = tl.trace(model, x, gradients_to_save="all", save_code_context=True)
loss = trace[trace.output_layers[0]].out.sum()
trace.log_backward(loss)
print(trace.summary(fields=["name", "shape", "params"]))

An `Op` is the most concrete record: one operation, one output tensor, and graph links to parents and children.

In [ ]:
op = trace[1]
print(f"record type: {type(op).__name__}")
print(f"layer label: {op.layer_label}")
print(f"function: {op.func_name}")
print(f"shape/dtype: {op.shape} / {op.dtype}")
print(f"parents -> children: {op.parents} -> {op.children}")
print(f"module call: {op.module}")
print(f"trace back-pointer is trace: {op.trace is trace}")

A `Layer` is the user-facing graph position. In this single-pass model each layer has one op, but the layer object is the stable place to inspect shape, module, params, and saved output.

In [ ]:
layer = trace.layers[op.layer_label]
print(f"record type: {type(layer).__name__}")
print(f"layer label: {layer.layer_label}")
print(f"ops in this layer: {list(layer.ops)}")
print(f"saved output shape: {tuple(layer.out.shape)}")
print(f"module call label: {layer.module}")
print(f"param addresses: {[param.address for param in layer.params]}")
print(f"trace back-pointer is trace: {layer.trace is trace}")

A `Module` aggregates all calls for one PyTorch module address. A `ModuleCall` is one concrete forward invocation of that module in this trace.

In [ ]:
for address in ["proj", "norm", "head"]:
    module = trace.modules[address]
    print(
        f"Module {module.address}: class={module.class_name} inputs={module.input_layers} outputs={module.output_layers}"
    )
    for position in range(len(module.calls)):
        call = module.calls[position]
        print(f"  call {call.call_label}: ops={list(call.ops)} outputs={call.output_layers}")

`Param` and `Buffer` records connect model state to graph use. Parameters expose trainability and shape; buffers expose non-parameter tensors such as running statistics or registered constants.

In [ ]:
print("Parameters:")
for param in trace.params:
    print(
        f"  {param.address:18s} shape={param.shape} trainable={param.trainable} "
        f"module={param.module_address} memory={param.memory_str}"
    )

print("\nBuffers:")
for buffer in trace.buffers:
    print(
        f"  {buffer.address:18s} shape={buffer.shape} module={buffer.module} "
        f"memory={buffer.memory_str}"
    )

After `log_backward`, TorchLens also exposes `GradFn` and `GradFnCall` records. These describe the PyTorch autograd graph and connect backward nodes back to forward ops when possible.

In [ ]:
print(f"grad fns: {len(trace.grad_fns)}")
print(f"grad fn calls: {len(trace.grad_fn_calls)}")
for grad_fn in list(trace.grad_fns)[:5]:
    print(
        f"  {grad_fn.label:18s} type={grad_fn.type:18s} "
        f"op_label={grad_fn.op_label} calls={grad_fn.num_calls}"
    )
for grad_call in list(trace.grad_fn_calls)[:3]:
    print(f"  call {grad_call.call_label}: grad_fn={grad_call.label} saved={grad_call.is_saved}")

Cross-references are the core of the data model: records point back to the parent `Trace`, and label fields let you resolve related records with `trace[...]`.

In [ ]:
output_label = trace.modules["head"].output_layers[0]
output_layer = trace[output_label]
print(f"head output layer label: {output_label}")
print(f"resolved layer type: {type(output_layer).__name__}")
print(f"resolved layer trace back-pointer: {output_layer.trace is trace}")
print(
    f"parents resolved through trace: {[trace[parent].layer_label for parent in output_layer.parents]}"
)

## 🔧 Sandbox

Pick any layer label from the trace and inspect its neighboring records.

In [ ]:
# TODO: try target_label = "relu_1_2", "layer_norm_1_3", or trace.output_layers[0].
target_label = trace.output_layers[0]
target = trace[target_label]
print(f"target: {target.layer_label} ({type(target).__name__})")
print(f"parents: {target.parents}")
print(f"children: {target.children}")
print(f"module: {target.module}")
print(f"shape: {target.shape}, memory: {target.memory_str}")